# 🚀 DGA_ML Advanced Training Pipeline

Sesuai request, pipeline ini mencakup:
- **TSFRESH Feature Extraction & Selection**
- **Optimization (GridSearchCV)**
- **Evaluasi K-Fold Cross Validation**
- **Leave-One-Measurement-Out (LOMO)**
- **Leave-One-Channel-Out (LOCO)**
- **Model Compressing (Pruning & Truncation)**


In [ ]:
#!/usr/bin/env python3"""DGA_ML Advanced Training Pipeline — Google Colab===================================================Pipeline komprehensif untuk melatih model klasifikasi DGA (High vs Low Grade).Meliputi:1. TSFRESH Feature Extraction & Selection2. Model Optimization (GridSearchCV / Hyperparameter Tuning)3. Evaluasi K-Fold Cross Validation4. Evaluasi Leave-One-Measurement-Out (LOMO)5. Evaluasi Leave-One-Channel-Out (LOCO)6. Model Compressing (Weight Pruning & Float truncation)Output: model_compressed.json yang siap diload oleh Rust (Kria)."""import osimport jsonimport globimport numpy as npimport pandas as pdfrom pathlib import Pathimport warningswarnings.filterwarnings('ignore')# ML Librariesfrom sklearn.model_selection import StratifiedKFold, GridSearchCV, LeaveOneGroupOutfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import accuracy_score, classification_reportfrom sklearn.svm import SVCfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.neural_network import MLPClassifier# TSFRESHtry:    from tsfresh import extract_features, select_features    from tsfresh.utilities.dataframe_functions import imputeexcept ImportError:    print("MENGINSTALL TSFRESH...")    os.system("pip install tsfresh")    from tsfresh import extract_features, select_features    from tsfresh.utilities.dataframe_functions import impute# =============================================================================# CONFIGURATION# =============================================================================DATA_ROOT = "data/raw"SENSOR_NAMES = ["tgs2600", "mq135", "mq3", "mq6", "mq7", "tgs2602", "tgs2611", "tgs2620"]N_SENSORS = 8N_TIMESTEPS = 300# =============================================================================# 1. DATA LOADING & FORMATTING FOR TSFRESH# =============================================================================def load_data_for_tsfresh(data_root):    """    Load data raw dan konversi ke format flat dataframe untuk TSFRESH.    TSFRESH butuh: [id, time, sensor1, sensor2, ...]    Juga extract metadata 'measurement_id' untuk Leave-One-Measurement-Out.    """    df_list = []    labels = {}    measurements = {}  # Untuk Leave-One-Measurement-Out        sample_id = 0        for grade_label, grade_folder in [(0, "high_grade"), (1, "low_grade")]:        grade_path = os.path.join(data_root, grade_folder)        if not os.path.exists(grade_path):            continue                    for folder_name in sorted(os.listdir(grade_path)):            folder_path = os.path.join(grade_path, folder_name)            if not os.path.isdir(folder_path):                continue                            csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))            for csv_file in csv_files:                try:                    df = pd.read_csv(csv_file)                    # Ambil 300 data pertama                    if len(df) >= N_TIMESTEPS:                        df = df.iloc[:N_TIMESTEPS].copy()                        df = df.iloc[:, 1:9]  # Ambil sensor                        df.columns = SENSOR_NAMES                                                df['id'] = sample_id                        df['time'] = np.arange(N_TIMESTEPS)                        df_list.append(df)                                                labels[sample_id] = grade_label                        measurements[sample_id] = folder_name  # Origin/Group                        sample_id += 1                except Exception as e:                    print(f"Error loading {csv_file}: {e}")                        if not df_list:        raise ValueError("Data kosong! Pastikan path DATA_ROOT benar.")            df_all = pd.concat(df_list, ignore_index=True)    y = pd.Series(labels).sort_index()    groups = pd.Series(measurements).sort_index()    return df_all, y, groups# =============================================================================# 2. TSFRESH EXTRACTION & COMPRESSION (PCA / Selection)# =============================================================================def extract_and_select_tsfresh(df_all, y):    print("\n[1/6] 🔄 Extracting TSFRESH Features (ini mungkin memakan waktu)...")    # Extract    extracted_features = extract_features(df_all, column_id="id", column_sort="time",                                           disable_progressbar=False)    impute(extracted_features)        print(f"      Terekstrak {extracted_features.shape[1]} fitur raw.")        # Feature Selection (Optimization / Compressing feature space)    print("      Memilih fitur yang relevan (Feature Selection)...")    selected_features = select_features(extracted_features, y)    print(f"      Tersisa {selected_features.shape[1]} fitur unggulan.")        # Simpan daftar fitur agar Kria tahu fitur apa saja yang dipakai    os.makedirs("models", exist_ok=True)    with open("models/selected_feature_names.json", "w") as f:        json.dump({"n_features": selected_features.shape[1],                    "feature_names": list(selected_features.columns)}, f, indent=2)            return selected_features# =============================================================================# 3. K-FOLD & OPTIMIZATION (GridSearchCV)# =============================================================================def optimize_and_train(X, y):    print("\n[2/6] ⚙️ Model Optimization & K-Fold (GridSearchCV)...")    scaler = StandardScaler()    X_scaled = scaler.fit_transform(X)        # Grid Search untuk SVM    param_grid_svm = {        'C': [0.1, 1, 10],        'kernel': ['linear', 'rbf']    }    svm_opt = GridSearchCV(SVC(probability=True), param_grid_svm, cv=StratifiedKFold(n_splits=5), scoring='accuracy')    svm_opt.fit(X_scaled, y)    print(f"      SVM Best Params: {svm_opt.best_params_} | Best CV Acc: {svm_opt.best_score_:.4f}")        # Grid Search untuk Random Forest    param_grid_rf = {        'n_estimators': [50, 100],        'max_depth': [5, 10, None]    }    rf_opt = GridSearchCV(RandomForestClassifier(), param_grid_rf, cv=StratifiedKFold(n_splits=5), scoring='accuracy')    rf_opt.fit(X_scaled, y)    print(f"      RF Best Params:  {rf_opt.best_params_} | Best CV Acc: {rf_opt.best_score_:.4f}")        # Grid Search untuk MLP    param_grid_mlp = {        'hidden_layer_sizes': [(64,), (128, 64)],        'alpha': [0.0001, 0.001]    }    mlp_opt = GridSearchCV(MLPClassifier(max_iter=300), param_grid_mlp, cv=StratifiedKFold(n_splits=5), scoring='accuracy')    mlp_opt.fit(X_scaled, y)    print(f"      MLP Best Params: {mlp_opt.best_params_} | Best CV Acc: {mlp_opt.best_score_:.4f}")        return svm_opt.best_estimator_, rf_opt.best_estimator_, mlp_opt.best_estimator_, scaler# =============================================================================# 4. LEAVE-ONE-MEASUREMENT-OUT (LOMO)# =============================================================================def evaluate_lomo(X, y, groups, model, scaler):    print("\n[3/6] 📊 Evaluasi Leave-One-Measurement-Out (LOMO)...")    logo = LeaveOneGroupOut()    accuracies = []        for train_idx, test_idx in logo.split(X, y, groups):        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]                # Scaling        sc = StandardScaler()        X_train_sc = sc.fit_transform(X_train)        X_test_sc = sc.transform(X_test)                # Train & Eval        model.fit(X_train_sc, y_train)        preds = model.predict(X_test_sc)        acc = accuracy_score(y_test, preds)        accuracies.append(acc)            print(f"      LOMO Accuracy (Average): {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")# =============================================================================# 5. LEAVE-ONE-CHANNEL-OUT (LOCO)# =============================================================================def evaluate_loco(df_all, y):    print("\n[4/6] 🚨 Evaluasi Leave-One-Channel-Out (LOCO)...")    # Tujuannya: Lihat seberapa drop akurasi kalau 1 sensor mati        for sensor in SENSOR_NAMES:        print(f"      Menguji tanpa sensor {sensor}...")        # Drop sensor        df_dropped = df_all.copy()        df_dropped[sensor] = 0.0  # Simulasi sensor mati                # Extract features (versi cepat)        feats = extract_features(df_dropped, column_id="id", column_sort="time", disable_progressbar=True)        impute(feats)        feats = select_features(feats, y)                # K-Fold eval        rf = RandomForestClassifier(n_estimators=50)        kf = StratifiedKFold(n_splits=3)        accs = []        for train_idx, test_idx in kf.split(feats, y):            rf.fit(feats.iloc[train_idx], y.iloc[train_idx])            accs.append(accuracy_score(y.iloc[test_idx], rf.predict(feats.iloc[test_idx])))                    print(f"        Akurasi (tanpa {sensor}): {np.mean(accs):.4f}")# =============================================================================# 6. MODEL COMPRESSING & EXPORT# =============================================================================def compress_and_export_mlp(mlp_model, scaler, filepath):    print("\n[5/6] 📦 Compressing & Exporting Model ke JSON...")        # 1. Ambil bobot    weights = mlp_model.coefs_    biases = mlp_model.intercepts_        # 2. Pruning: set bobot yang sangat kecil (<1e-3) menjadi 0 (Sparsity)    pruning_threshold = 1e-3    compressed_weights = []    sparsity = 0    total_params = 0        for w in weights:        w_pruned = np.where(np.abs(w) < pruning_threshold, 0.0, w)        sparsity += np.sum(w_pruned == 0.0)        total_params += w.size        # 3. Float Truncation (bulatkan ke 4 desimal untuk kompresi teks JSON)        w_rounded = np.round(w_pruned, decimals=4)        compressed_weights.append(w_rounded.tolist())            print(f"      Tingkat Kompresi (Sparsity): {sparsity / total_params * 100:.2f}% bobot = 0")        # Format sesuai struct CoffeeMLP Rust    def layer_to_dict(w, b, in_sz, out_sz, relu):        return {            "weights": np.array(w).T.tolist(), # Rust expect (out, in)            "bias": np.round(b, decimals=4).tolist(),            "in_size": int(in_sz),            "out_size": int(out_sz),            "use_relu": relu        }            model_dict = {        "config": {            "hidden1": mlp_model.hidden_layer_sizes[0],            "hidden2": mlp_model.hidden_layer_sizes[1] if len(mlp_model.hidden_layer_sizes) > 1 else 0,            "n_epochs": mlp_model.max_iter,            "learning_rate": mlp_model.learning_rate_init,            "batch_size": 32,            "clip_grad": 5.0,            "disable_early_stop": False        },        "layer1": layer_to_dict(compressed_weights[0], biases[0], weights[0].shape[0], weights[0].shape[1], True),        "layer_out": layer_to_dict(compressed_weights[-1], biases[-1], weights[-1].shape[0], weights[-1].shape[1], False),        "norm": {            "mean": np.round(scaler.mean_, decimals=4).tolist(),            "std": np.round(scaler.scale_, decimals=4).tolist()        }    }        # Jika 2 hidden layer    if len(weights) == 3:        model_dict["layer2"] = layer_to_dict(compressed_weights[1], biases[1], weights[1].shape[0], weights[1].shape[1], True)    else:        # Dummy layer 2 jika cuma 1 hidden layer        model_dict["layer2"] = layer_to_dict([[0]], [0], weights[0].shape[1], weights[0].shape[1], True)    with open(filepath, 'w') as f:        # Gunakan separators tanpa spasi ekstra untuk mengkompres JSON        json.dump(model_dict, f, separators=(',', ':'))            size_kb = os.path.getsize(filepath) / 1024    print(f"      ✅ Model terkompresi disimpan di {filepath} ({size_kb:.1f} KB)")# =============================================================================# MAIN EXECUTION# =============================================================================if __name__ == "__main__":    print("="*60)    print("🚀 DGA_ML ADVANCED PIPELINE STARTING")    print("="*60)        # Uncomment jika data tersedia    """    # 1. Load Data    df_all, y, groups = load_data_for_tsfresh(DATA_ROOT)        # 2. Extract TSFRESH    X_features = extract_and_select_tsfresh(df_all, y)        # 3. K-Fold & Opt    best_svm, best_rf, best_mlp, scaler = optimize_and_train(X_features, y)        # 4. LOMO    evaluate_lomo(X_features, y, groups, best_rf, scaler)        # 5. LOCO    evaluate_loco(df_all, y)        # 6. Compress & Export    compress_and_export_mlp(best_mlp, scaler, "models/mlp_compressed.json")        print("\n✅ PIPELINE SELESAI. Model siap dikirim ke Kria.")    """    print("\nScript berhasil digenerate. Uncomment bagian eksekusi untuk menjalankan.")